# Part 2: Modeling & Evaluation

This section performs ridge regression modeling and evaluation on fMRI response data using the custom embeddings generated from our pre-trained encoder (trained via masked language modeling in Part 1). As in Lab 3.1, we aim to predict voxel-wise BOLD responses based on linguistic representations, but in this lab, we compare the performance of our custom-trained embeddings to that of pre-trained embeddings from previous work.

## Objectives
- Use the embeddings generated by our self-trained encoder to predict fMRI voxel responses via ridge regression.
- Follow the same modeling pipeline as in Lab 3.1 to ensure comparability.
- Evaluate the model using the following metrics:
    - Mean test correlation coefficient (CC)
    - Median test CC
    - Top 1 percentile CC
    - Top 5 percentile CC
- For the best-performing embedding:
    - Visualize the distribution of correlation coefficients across voxels.
    - Analyze whether the model performs uniformly across the brain, and discuss implications under the PCS (Predictability, Computability, Stability) framework.
    - Perform a stability analysis across different subjects or stories.

## Files Used
This script assumes that the following data file exists in `../data/`:
- `X_lagged_mlm.joblib`: embeddings generated from our masked language modeling encoder, with temporal alignment and lag applied

Response data is assumed to be located under the following directory in the PSC computing environment:

`../../tmp_ondemand_ocean_mth240012p_symlink/shared/data/`

Each subject folder contains `.npy` files corresponding to voxel responses for individual stories. We also assume that ridge regression utilities are available through the provided `ridge_utils/` directory.


In [1]:
# Import necessary libraries

import numpy as np
import joblib
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
import sys
import importlib.util
import os

In [2]:
# Load the ridge.py module
embedding_dir = Path("../data")
subject2_dir = Path("../../tmp_ondemand_ocean_mth240012p_symlink/shared/data/subject2")
subject3_dir = Path("../../tmp_ondemand_ocean_mth240012p_symlink/shared/data/subject3")
ridge_py_path = Path("ridge_utils/ridge.py")
ridge_utils_dir = Path("ridge_utils") 

sys.path.append(str(ridge_utils_dir))

spec = importlib.util.spec_from_file_location("ridge", ridge_py_path)
ridge = importlib.util.module_from_spec(spec)
sys.modules["ridge"] = ridge
spec.loader.exec_module(ridge)
bootstrap_ridge = ridge.bootstrap_ridge

# Load the data
X_mlm = joblib.load(embedding_dir / "X_lagged_mlm.joblib")

def load_subject_data(subject_dir):
    return {f.stem: np.load(f) for f in subject_dir.glob("*.npy")}

X_dict = {"MaskLM": X_mlm}

# Load Response Data
Y_subject2 = load_subject_data(subject2_dir)
Y_subject3 = load_subject_data(subject3_dir)


In [3]:
"""
Performs bootstrap ridge regression across multiple word embeddings (BoW, Word2Vec, GloVe)
and two subjects (subject2, subject3) for each shared story. The workflow includes:

1. Iterating through each combination of embedding type and subject.
2. For each story that exists in both the embedding and subject's response data:
   - Verifies timepoint alignment between X (stimuli) and Y (response).
   - Applies z-score normalization to both X and Y.
   - Splits the data into 80% training and 20% test set.
   - Runs bootstrap ridge regression to compute voxel-level prediction performance.
   - Stores mean correlation coefficient (CC) across all voxels.
3. Handles shape mismatches and any runtime exceptions gracefully with logging.
4. Aggregates results into a DataFrame and saves them to a CSV file (`ridge_results.csv`).

Results:
    - A CSV file containing the mean correlation coefficient per (embedding, subject, story) triplet.
    - Enables later analysis of which embedding performs best across stories and subjects.
"""

from time import time
import pandas as pd

# Initialize variables
total_tasks = sum(len(X_dict[x_key]) for x_key in X_dict for _ in [0, 1])
task_counter = 0

all_corrs = []
results = []

# Function to standardize the data
def zs(v):
    std = v.std(axis=0)
    std[std == 0] = 1
    return (v - v.mean(axis=0)) / std

# Conduct bootstrap ridge regression
for x_key, x_data in X_dict.items():
    for subj_key, y_data in [("subj2", Y_subject2), ("subj3", Y_subject3)]:
        shared_stories = sorted(list(set(x_data.keys()) & set(y_data.keys())))
        for story in shared_stories: 
            print(f"\n[Running] {x_key} - {subj_key} - {story}")
            X = x_data[story]
            Y = y_data[story]

            if X.shape[0] != Y.shape[0]:
                print(f"  [Skip] Mismatch in shape: X={X.shape}, Y={Y.shape}")
                continue

            try:
                X_z = zs(X)
                Y_z = zs(Y)
                X_train, X_test, Y_train, Y_test = train_test_split(X_z, Y_z, test_size=0.2, random_state=42)
                alphas = np.logspace(0, 3, 20)
                start_time = time()
                wt, corrs, valalphas, _, _ = bootstrap_ridge(
                    X_train, Y_train, X_test, Y_test,
                    alphas=alphas, nboots=10, chunklen=10, nchunks=2,
                    return_wt=True
                )
                elapsed = time() - start_time
                print(f"  [Done] Mean CC={np.mean(corrs):.3f}, Time={elapsed:.1f}s")

                all_corrs.extend(corrs)  # accumulate all voxel-level CCs
                results.append((x_key, subj_key, story, float(np.mean(corrs))))

            except Exception as e:
                print(f"  [Error] {story}: {e}")

df = pd.DataFrame(results, columns=["X_type", "subject", "story", "mean_cc"])
df.to_csv("ridge_results.csv", index=False)


[Running] MaskLM - subj2 - adollshouse
  [Done] Mean CC=0.027, Time=8.8s

[Running] MaskLM - subj2 - adventuresinsayingyes
  [Done] Mean CC=0.011, Time=9.6s

[Running] MaskLM - subj2 - afatherscover
  [Done] Mean CC=0.028, Time=8.9s

[Running] MaskLM - subj2 - afearstrippedbare
  [Done] Mean CC=-0.003, Time=10.3s

[Running] MaskLM - subj2 - againstthewind
  [Done] Mean CC=0.038, Time=7.5s

[Running] MaskLM - subj2 - alternateithicatom
  [Done] Mean CC=0.007, Time=9.1s

[Running] MaskLM - subj2 - avatar
  [Done] Mean CC=0.044, Time=9.5s

[Running] MaskLM - subj2 - backsideofthestorm
  [Done] Mean CC=-0.006, Time=9.2s

[Running] MaskLM - subj2 - becomingindian
  [Done] Mean CC=0.032, Time=9.7s

[Running] MaskLM - subj2 - beneaththemushroomcloud
  [Done] Mean CC=0.028, Time=9.3s

[Running] MaskLM - subj2 - birthofanation
  [Done] Mean CC=0.025, Time=8.4s

[Running] MaskLM - subj2 - bluehope
  [Done] Mean CC=0.003, Time=9.9s

[Running] MaskLM - subj2 - breakingupintheageofgoogle
  [Done] 

/jet/home/jlee45/.conda/envs/env_214/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


  [Done] Mean CC=0.042, Time=10.3s

[Running] MaskLM - subj2 - catfishingstrangerstofindmyself
  [Done] Mean CC=0.012, Time=8.9s

[Running] MaskLM - subj2 - cautioneating
  [Done] Mean CC=0.022, Time=8.5s

[Running] MaskLM - subj2 - christmas1940
  [Done] Mean CC=0.044, Time=8.2s

[Running] MaskLM - subj2 - cocoonoflove
  [Done] Mean CC=0.013, Time=10.1s

[Running] MaskLM - subj2 - comingofageondeathrow
  [Done] Mean CC=0.033, Time=9.1s

[Running] MaskLM - subj2 - escapingfromadirediagnosis
  [Done] Mean CC=0.036, Time=9.2s

[Running] MaskLM - subj2 - exorcism
  [Done] Mean CC=0.056, Time=10.7s

[Running] MaskLM - subj2 - eyespy
  [Done] Mean CC=0.028, Time=9.7s

[Running] MaskLM - subj2 - findingmyownrescuer
  [Done] Mean CC=0.027, Time=9.4s

[Running] MaskLM - subj2 - firetestforlove
  [Done] Mean CC=0.036, Time=8.8s

[Running] MaskLM - subj2 - food
  [Done] Mean CC=0.026, Time=9.4s

[Running] MaskLM - subj2 - forgettingfear
  [Done] Mean CC=0.014, Time=8.1s

[Running] MaskLM - subj2

In [4]:
"""
Summary statistics of voxel-wise correlation coefficients (CCs):
- Mean CC provides an overall average performance across all voxels.
- Median CC represents the middle value, giving a robust sense of central tendency.
- Top 5% Quantile (95th percentile) shows the threshold above which the top 5% of voxel performances lie.
- Top 1% Quantile (99th percentile) highlights the very best-performing voxels in the distribution.
"""

print("Mean CC:", np.mean(all_corrs))
print("Median CC:", np.median(all_corrs))
print("Top 5% Quantile:", np.quantile(all_corrs, 0.95))
print("Top 1% Quantile:", np.quantile(all_corrs, 0.99))

Mean CC: 0.02469962796815975
Median CC: 0.023452006016043053
Top 5% Quantile: 0.24476113135945526
Top 1% Quantile: 0.3444113304324249
